# Capstone — Choosing the Right Tool and Stating Honest Limits

**DS4DH · Module 12 — Advanced Synthesis**

*Technique:* Method selection and writing the scope of valid inference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/12_synthesis_capstone.ipynb)

Data: `merged_dataset.csv`, `city_summary.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

# This notebook reads the CSVs sitting next to it. In Colab, upload them from
# the pack's data/ folder when prompted. The exists() guard means a re-run
# part-way through a session will not ask you to upload all over again.
NEEDED = ['merged_dataset.csv', 'city_summary.csv']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files
        print('Upload from the data/ folder of the pack: ' + ', '.join(missing))
        files.upload()
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))

df       = pd.read_csv('merged_dataset.csv')
df_city  = pd.read_csv('city_summary.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Synthesis is choosing, not accumulating. A report that runs every method on every
variable is not thorough; it is undirected, and it invites the reader to pick the
result they like.

This capstone does three things:

1. reproduces the whole course's findings in one pass, so you can see them together
2. gives you a decision procedure for picking a method from a question
3. asks you to write the scope statement that bounds all of it

## The decision tree

| Your question | Method | Module |
|---|---|---|
| What does this look like? | descriptive stats, distributions | 02 |
| Do two groups differ? | paired comparison + t-test + effect size | 02, 04 |
| Is it just noise? | Bonferroni / Holm correction | 04 |
| Is it explained by something else? | OLS with fixed effects | 05 |
| Are my standard errors trustworthy? | HC1 robust SEs | 05 |
| Are there groups nobody labelled? | K-Means + Ward, cross-validated | 06 |
| What predicts this best? | gradient boosting + held-out scoring | 07 |
| What does the model rely on? | permutation importance | 07 |
| I need a number that does not exist | composite index + weight sensitivity | 08 |
| Where is it concentrated? | ranking, grouping, CV of dispersion | 09 |
| Does my unit choice drive the answer? | MAUP check | 09 |
| How do I show it? | small multiples, shared axes, reference line | 10 |
| Should anyone act on it? | three-hurdle classification | 11 |

The tree runs top to bottom. Reaching for gradient boosting before you have
described the distribution is the most common way to waste a week.

In [ ]:
# One pass over the whole course.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & csd['cma'].isin(CITIES)].copy()

imm = csd[csd['immigrant_status'] == 'Immigrant'][['csd_code', 'cma', 'Renter']]
nim = csd[csd['immigrant_status'] == 'Non-immigrants'][['csd_code', 'Renter']]
pen = imm.merge(nim, on='csd_code', suffixes=('_imm', '_nim')).dropna()
pen = pen[pen['cma'].isin(CITIES)]
pen['penalty'] = pen['Renter_imm'] - pen['Renter_nim']

print(f'CSDs in four CMAs                : {len(base)}')
print(f'  with complete burden data      : {base["Total"].notna().sum()}')
print(f'  usable for paired comparison   : {len(pen)}')

In [ ]:
# Finding 1 (M02) — descriptive.
d = base.dropna(subset=['Total'])
print('FINDING 1 — housing burden by CMA (descriptive)')
for city in CITIES:
    s = d[d['cma'] == city]['Total']
    print(f'  {city:<12} median {s.median():>5.1f}%   IQR {s.quantile(.25):.1f}-{s.quantile(.75):.1f}   n={len(s)}')
print()

# Finding 2 (M04) — inference.
print('FINDING 2 — immigrant renter gap, corrected for 4 tests')
BONF = 0.05 / 4
for city in CITIES:
    s = pen[pen['cma'] == city]['penalty']
    t, p = stats.ttest_1samp(s, 0)
    dd = s.mean() / s.std(ddof=1)
    mark = 'SURVIVES' if p < BONF else '-'
    print(f'  {city:<12} {s.mean():>+6.2f}pp  p={p:<8.4f} d={dd:>+6.2f}  {mark}')

In [ ]:
# Finding 3 (M05) — regression.
reg = csd[csd['immigrant_status'].isin(['Immigrant', 'Non-immigrants'])
          & csd['cma'].isin(CITIES)].dropna(subset=['Renter']).copy()
reg['is_immigrant'] = (reg['immigrant_status'] == 'Immigrant').astype(int)
dums = pd.get_dummies(reg['cma'], drop_first=True, dtype=float)

m_a = sm.OLS(reg['Renter'], sm.add_constant(reg[['is_immigrant']])).fit()
m_b = sm.OLS(reg['Renter'],
             sm.add_constant(pd.concat([reg[['is_immigrant']], dums], axis=1))
             ).fit(cov_type='HC1')
ci = m_b.conf_int().loc['is_immigrant']

print('FINDING 3 — pooled gap, before and after controlling for city')
print(f'  no controls        {m_a.params["is_immigrant"]:>+6.2f}pp')
print(f'  + city fixed FX    {m_b.params["is_immigrant"]:>+6.2f}pp  '
      f'95% CI [{ci[0]:+.2f}, {ci[1]:+.2f}]  p={m_b.pvalues["is_immigrant"]:.3f}')
print(f'  -> {(1 - m_b.params["is_immigrant"] / m_a.params["is_immigrant"]):.0%} of the raw gap was city composition')

In [ ]:
# Finding 4 (M09) — within-city inequality.
print('FINDING 4 — dispersion within each CMA')
for city in CITIES:
    s = d[d['cma'] == city]['Total']
    p10, p90 = s.quantile([0.1, 0.9])
    print(f'  {city:<12} CV={s.std() / s.mean():.3f}   p90/p10={p90 / p10:.2f}')
print()
print('FINDING 5 — the comparison that matters')
between = d.groupby('cma', observed=True)['Total'].mean()
print(f'  spread of city means      : {between.max() - between.min():.1f} pp')
print(f'  spread within Montréal    : {d[d["cma"] == "Montréal"]["Total"].quantile(.9) - d[d["cma"] == "Montréal"]["Total"].quantile(.1):.1f} pp')
print('  Variation WITHIN cities exceeds variation BETWEEN them.')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
groups = [d[d['cma'] == c]['Total'] for c in CITIES]
ax.boxplot(groups)
ax.set_xticklabels(CITIES)
ax.axhline(30, color='#E8663D', ls='--', lw=1.5, label='30% affordability line')
ax.set_ylabel('Total STIR (%)')
ax.set_title('The one chart for this analysis: within-city spread dwarfs between-city difference')
ax.legend()
plt.tight_layout()
plt.show()

## Honest limits

Four bounds apply to everything above. They are not hedging — each one rules out
a specific sentence someone will otherwise write.

1. **One snapshot.** No claim about trends, worsening, or change. Rules out
   "housing is becoming less affordable".
2. **Four CMAs.** Montréal, Toronto, Edmonton, Vancouver only. Rules out
   "in Canada".
3. **Suppression is not random.** Small municipalities are systematically absent.
   Rules out "across municipalities" without qualification.
4. **Association, not causation.** Nothing here is a controlled comparison. Rules
   out "because of", "leads to", "the effect of".

In [ ]:
claims = [
    ('Housing burden is rising in Canadian cities',
     False, 'fails limits 1 and 2 — one snapshot, four CMAs'),
    ('Within these four CMAs, housing burden varies more between municipalities '
     'than between cities',
     True, 'supported by Finding 5'),
    ('Immigrant renters pay more of their income on housing',
     False, 'fails — the pooled gap is not distinguishable from zero'),
    ('In Edmonton, immigrant renters in comparable subdivisions spend a smaller '
     'share of income on shelter than non-immigrant renters',
     True, 'supported by Finding 2, the only result surviving correction'),
    ('Being an immigrant causes lower housing burden in Edmonton',
     False, 'fails limit 4 — association, not causation'),
]

for text, ok, why in claims:
    print(f'{"SUPPORTED" if ok else "NOT SUPPORTED":<15} {text}')
    print(f'{"":<15} {why}')
    print()

### 🔧 Your turn 1

Add two claims of your own to the list — one you believe is supported and one you
believe is not — and justify each against the four limits.

The skill being tested is not caution. It is knowing exactly which limit rules out
which sentence.

### 🔧 Your turn 2

Retrieve the scope statement you wrote in notebook 01b, before you had run any of
this.

Compare it to what you would write now. What did you not know to exclude? That
gap is the most useful thing to carry into your own project.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** A supported claim in the right register:

> Among the four metropolitan areas studied, more than a quarter of census
> subdivisions with reported data show total shelter-cost-to-income ratios above
> the 30% affordability threshold.

An unsupported one that looks reasonable:

> Municipalities with lower incomes have higher housing burden because residents
> cannot access owner-occupied housing.

The first is a count within a stated sample. The second smuggles a mechanism —
"because residents cannot access" — that nothing in the data tests, failing limit
4 even though the correlation it rests on is real.

**Your turn 2.** The usual gap is that the early scope statement excludes time and
geography (the obvious limits) but not suppression bias or the ratio-of-ratios
problem from 09b. Those are the ones you only learn by working with the data.
Writing the statement early and revising it late is the practice worth keeping —
the revision is where the methodological learning actually shows up.

</details>

## The one-page brief

You now have everything needed for the deliverable this course has been building
toward:

- **Finding** — within-city variation exceeds between-city variation; one
  robust group difference, in Edmonton, running opposite to expectation
- **Method** — paired within-CSD comparison, Bonferroni-corrected, checked against
  OLS with city fixed effects and HC1 standard errors
- **Uncertainty** — intervals reported, weighting sensitivity tested
- **Limits** — the four bounds above, stated rather than implied
- **Recommendation** — geographic rather than group-based targeting; no programme
  sized on the pooled estimate

That is the whole course. The methods were the means; this page is the output.